In [1]:
import pandas as pd

data = pd.read_csv("../../data/train-2025.csv", low_memory=False)
len(data)

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


1023552

In [2]:
groups = {
    "1 Identity & attributes": ["id", "stmt_date", "HQ_city", "legal_struct", "ateco_sector", "fs_year", "def_date"],
    "2 Assets": ["asst_intang_fixed", "asst_tang_fixed", "asst_fixed_fin", "asst_current", "AR", "cash_and_equiv", "asst_tot", "goodwill"],
    "3 Equity & liabilities": ["eqty_tot", "eqty_corp_family_tot", "liab_lt", "liab_lt_emp", "debt_bank_st", "debt_bank_lt",
                "debt_fin_st", "debt_fin_lt", "AP_st", "AP_lt", "debt_st", "debt_lt"],
    "4 Income statement": ["rev_operating", "COGS", "prof_operations", "inc_financing", "exp_financing", "prof_financing",
              "inc_extraord", "taxes", "profit", "ebitda"],
    "5 Cash flow & derived ratios": ["cf_operations", "wc_net", "margin_fin", "days_rec", "roa", "roe"],
}
assert sorted(sum(groups.values(), [])) == sorted(data.columns.drop("Unnamed: 0"))

rows = []
for g, cols in groups.items():
    for c in cols:
        rows.append({"group": g, "col": c, "n_missing": data[c].isna().sum(),
                     "pct_missing": round(data[c].isna().mean() * 100, 2),
                     "n_zero": (data[c] == 0).sum(), "pct_zero": round((data[c] == 0).mean() * 100, 2)})
miss = pd.DataFrame(rows).set_index(["group", "col"])
miss

n_missing  pct_missing  \
group                        col                                            
1 Identity & attributes      id                            0         0.00   
                             stmt_date                     0         0.00   
                             HQ_city                     277         0.03   
                             legal_struct                  0         0.00   
                             ateco_sector                  0         0.00   
                             fs_year                       0         0.00   
                             def_date                 978827        95.63   
2 Assets                     asst_intang_fixed           162         0.02   
                             asst_tang_fixed             103         0.01   
                             asst_fixed_fin              237         0.02   
                             asst_current                  1         0.00   
                             AR                           12         0.00   
                             cash_and_equiv               17         0.00   
                             asst_tot                      0         0.00   
                             goodwill                     21         0.00   
3 Equity & liabilities       eqty_tot                      1         0.00   
                             eqty_corp_family_tot    1023552       100.00   
                             liab_lt                     294         0.03   
                             liab_lt_emp                 162         0.02   
                             debt_bank_st             121331        11.85   
                             debt_bank_lt              90971         8.89   
                             debt_fin_st              122772        11.99   
                             debt_fin_lt               92068         8.99   
                             AP_st                    120635        11.79   
                             AP_lt                     92244         9.01   
                             debt_st                       5         0.00   
                             debt_lt                     155         0.02   
4 Income statement           rev_operating               174         0.02   
                             COGS                         24         0.00   
                             prof_operations              21         0.00   
                             inc_financing                26         0.00   
                             exp_financing               367         0.04   
                             prof_financing              656         0.06   
                             inc_extraord                305         0.03   
                             taxes                        32         0.00   
                             profit                       20         0.00   
                             ebitda                       81         0.01   
5 Cash flow & derived ratios cf_operations                88         0.01   
                             wc_net                        6         0.00   
                             margin_fin                40758         3.98   
                             days_rec                 740211        72.32   
                             roa                          27         0.00   
                             roe                       72937         7.13   

                                                   n_zero  pct_zero  
group                        col                                     
1 Identity & attributes      id                         0      0.00  
                             stmt_date                  0      0.00  
                             HQ_city                    0      0.00  
                             legal_struct               0      0.00  
                             ateco_sector               0      0.00  
                             fs_year                    0      0.00  
                         

In [3]:
dbt = ["debt_bank_st", "debt_bank_lt", "debt_fin_st", "debt_fin_lt", "AP_st", "AP_lt"]
data[dbt].isna().value_counts().head(10)

debt_bank_st  debt_bank_lt  debt_fin_st  debt_fin_lt  AP_st  AP_lt
False         False         False        False        False  False    889407
True          True          True         True         True   True      79251
              False         True         False        True   False     41245
False         True          False        True         False  True      11200
              False         True         True         False  True        821
True          False         True         False        False  False       295
False         False         True         False        False  False       293
              True          True         True         False  True        209
True          True          True         True         False  True        184
              False         True         True         False  True        166
Name: count, dtype: int64

In [4]:
# Missing-value handling; see docs/project-update/features/02-missing-value.md
DROP_COLS = ["Unnamed: 0", "eqty_corp_family_tot", "days_rec", "roe"]
NON_NUMERIC = ["stmt_date", "legal_struct", "def_date"]
DEBT_DETAIL = ["debt_bank_st", "debt_bank_lt", "debt_fin_st", "debt_fin_lt", "AP_st", "AP_lt"]
# Method B: columns filled with (sector median of column / asst_tot) * asst_tot
B_COLS = [
    "asst_intang_fixed", "asst_tang_fixed", "asst_fixed_fin", "asst_current", "AR", "cash_and_equiv", "goodwill",
    "eqty_tot", "liab_lt", "liab_lt_emp", *DEBT_DETAIL, "debt_st", "debt_lt",
    "rev_operating", "COGS", "prof_operations", "inc_financing", "exp_financing", "prof_financing",
    "inc_extraord", "taxes", "profit", "ebitda", "cf_operations",
]


def sector_medians(train):
    """Median table for method B, computed from the training data: indexed by ateco_sector, last row "ALL" is the overall median.
    The asst_tot column holds the median amount (safeguard for holdout rows missing asst_tot); the other columns hold median ratios to asst_tot."""
    train = _to_numeric(train)
    ratio = train[B_COLS].div(train["asst_tot"], axis=0)
    ratio["asst_tot"] = train["asst_tot"]
    med = ratio.groupby(train["ateco_sector"]).median()
    med.loc["ALL"] = ratio.median()
    return med


def _to_numeric(df):
    # Blanks, "NA" and other missing-value spellings all become NaN
    df = df.copy()
    for c in df.columns.difference(NON_NUMERIC):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def fill_missing(df, medians):
    """Take raw data, return filled data. medians must be the result of sector_medians(training data)."""
    df = _to_numeric(df).drop(columns=DROP_COLS, errors="ignore")

    # Method A: a missing HQ_city becomes its own category
    df["HQ_city"] = df["HQ_city"].map("{:.0f}".format, na_action="ignore").fillna("Unknown")

    # Method C: record the flag before filling
    df["debt_detail_missing"] = df[DEBT_DETAIL].isna().any(axis=1).astype(int)

    # Method D: prof_operations = rev_operating - COGS; restore whichever of the three is missing
    df["rev_operating"] = df["rev_operating"].fillna(df["prof_operations"] + df["COGS"])
    df["COGS"] = df["COGS"].fillna(df["rev_operating"] - df["prof_operations"])
    df["prof_operations"] = df["prof_operations"].fillna(df["rev_operating"] - df["COGS"])

    # Method B: look up the sector medians; sectors absent from training use the overall median
    m = medians.drop(index="ALL").reindex(df["ateco_sector"]).set_axis(df.index).fillna(medians.loc["ALL"])
    df["asst_tot"] = df["asst_tot"].fillna(m["asst_tot"])
    df[B_COLS] = df[B_COLS].fillna(m[B_COLS].mul(df["asst_tot"], axis=0))

    # Method E: restore by formula (after B and D, so every input is already filled)
    fixed = df["asst_intang_fixed"] + df["asst_tang_fixed"] + df["asst_fixed_fin"]
    df["margin_fin"] = df["margin_fin"].fillna(df["eqty_tot"] - fixed)
    df["wc_net"] = df["wc_net"].fillna(df["asst_current"] - df["debt_st"])
    df["roa"] = df["roa"].fillna(df["prof_operations"] / df["asst_tot"] * 100)

    return df


In [5]:
medians = sector_medians(data)
filled = fill_missing(data, medians)

# Check: no missing values except def_date; existing values are untouched; flag counts match the doc
assert filled.drop(columns="def_date").isna().sum().sum() == 0
num = data.columns.difference(DROP_COLS + NON_NUMERIC + ["HQ_city"])
orig = data[num].notna()
assert (filled[num][orig] == data[num][orig]).sum().sum() == orig.sum().sum()
assert filled["debt_detail_missing"].sum() == 134145
assert (filled["HQ_city"] == "Unknown").sum() == 277

# Check: missing-value spellings a holdout may use, and a sector never seen in training
raw = data.head(3).astype(str)
raw.loc[raw.index[0], ["AR", "HQ_city", "asst_tot"]] = ["", " ", "NA"]
raw.loc[raw.index[1], "ateco_sector"] = "999"
out = fill_missing(raw, medians)
assert out.drop(columns="def_date").isna().sum().sum() == 0
assert out["HQ_city"].iloc[0] == "Unknown"
filled.shape


(1023552, 41)